In [62]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import  pandas as pd 
import json 
import os
from glob import glob
import seaborn as sns 
import numpy as np 
import re
import tikzplotly
import plotly.express as px
from IPython.display import display
from PIL import Image
import matplotlib as mpl
import matplotlib.pyplot as plt 
import plotly 
import plotly.graph_objects as go

from IPython.display import IFrame

from utils.benchmark import * 

In [63]:
notebook_name="01-c5.large-modubft-spread-over-azs"
os.makedirs(f"outputs/{notebook_name}", exist_ok=True)
 

In [64]:
# folder="../../aws/benchmark/out/modubft/full_spread_modubft_with_bytes_sent/" 
folder = "../../aws/benchmark/out/modubft/20251025-215157-large/"
os.listdir(folder)

['20251025215534-cb1-v512',
 '20251025215609-cb1-v4096',
 '20251025220150-cb1-v512',
 '20251025220231-cb1-v4096',
 '20251025220833-cb1-v512',
 '20251025220922-cb1-v4096',
 '20251025221633-cb1-v512',
 '20251025221731-cb1-v4096',
 '20251025222547-cb1-v512',
 '20251025222646-cb1-v4096',
 'modubft_peers_12.txt',
 'modubft_peers_18.txt',
 'modubft_peers_21.txt',
 'modubft_peers_3.txt',
 'modubft_peers_6.txt']

In [65]:
benchmarks = glob(f"{folder}*/")
benchmarks

['../../aws/benchmark/out/modubft/20251025-215157-large/20251025215534-cb1-v512/',
 '../../aws/benchmark/out/modubft/20251025-215157-large/20251025215609-cb1-v4096/',
 '../../aws/benchmark/out/modubft/20251025-215157-large/20251025220150-cb1-v512/',
 '../../aws/benchmark/out/modubft/20251025-215157-large/20251025220231-cb1-v4096/',
 '../../aws/benchmark/out/modubft/20251025-215157-large/20251025220833-cb1-v512/',
 '../../aws/benchmark/out/modubft/20251025-215157-large/20251025220922-cb1-v4096/',
 '../../aws/benchmark/out/modubft/20251025-215157-large/20251025221633-cb1-v512/',
 '../../aws/benchmark/out/modubft/20251025-215157-large/20251025221731-cb1-v4096/',
 '../../aws/benchmark/out/modubft/20251025-215157-large/20251025222547-cb1-v512/',
 '../../aws/benchmark/out/modubft/20251025-215157-large/20251025222646-cb1-v4096/']

In [66]:
c5axlarge_folder="../../aws/benchmark/out/modubft/20251022-011614/"
benchmarks = benchmarks + glob(f"{c5axlarge_folder}*/")

In [67]:


throughputs = process_throughput_benchmarks(benchmarks)

In [68]:
throughputs["benchmark"] = throughputs["benchmark"].apply(lambda s: "c5a.large" if "large" in s else "c5a.xlarge")

xs = []
ys = []
cats = [] 

for (group,df) in throughputs.groupby(["benchmark","vallen"]): 
    cats.append(f"{group[0]}-{group[1]}B")
    xs.append(df["num_peers"].tolist())
    ys.append(df["throughput"].tolist())
    

tikz_plot = TikzPlotGenerator(
    xs=xs,
    ys=ys,
    cat=cats,
    xlabel="Number of Peers",
    ylabel="Throughput (ops/sec)",
    filename=f"outputs/{notebook_name}/modubft-spread-over-azs-throughput.tex"
)
tikz_plot.save()
tikz_plot.compile(output_dir=f"outputs/{notebook_name}/")


[np.int64(3), np.int64(6), np.int64(12), np.int64(18), np.int64(21)]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode

(./outputs/01-c5.large-modubft-spread-over-azs/modubft-spread-over-azs-throughp
ut.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-dist

0

In [69]:


processed_cpu_usage = process_cpu_usage(benchmarks)


for role in [2]:
    df_role = processed_cpu_usage[processed_cpu_usage["role"] == role].copy()
    df_role["benchmark"] = df_role["benchmark"].apply(
        lambda s: "c5a.large" if "large" in s else "c5a.xlarge"
    )

    xs = []
    ys = []
    cats = []

    for (group, df) in df_role.groupby(["benchmark", "vallen"]):
        cats.append(f"{group[0]}-{group[1]}B")
        sorteddf = df.sort_values(["num_peers"])
        xs.append(sorteddf["num_peers"].tolist())
        ys.append(sorteddf["cpu_usage"].tolist())

    tikz_plot_cpu = TikzPlotGenerator(
        xs=xs,
        ys=ys,
        cat=cats,
        xlabel="Number of Peers",
        ylabel="CPU Usage",
        filename=f"outputs/{notebook_name}/modubft-spread-over-azs-cpu-usage-role{role}.tex"
    )
    tikz_plot_cpu.save()
    tikz_plot_cpu.compile(output_dir=f"outputs/{notebook_name}/")


for role in [1]:
    df_role = processed_cpu_usage[processed_cpu_usage["role"] == role].copy()
    df_role["benchmark"] = df_role["benchmark"].apply(
        lambda s: "c5a.large" if "large" in s else "c5a.xlarge"
    )
    df_role = [df_role.groupby(["benchmark","vallen","num_peers"]).max().reset_index().assign(agg="max"),
    df_role.groupby(["benchmark","vallen","num_peers"]).min().reset_index().assign(agg="min")]
    df_role = pd.concat(df_role)
    xs = []
    ys = []
    cats = []

    for (group, df) in df_role.groupby(["benchmark", "vallen","agg"]):
        sorteddf = df.sort_values(["num_peers"])
        cats.append(f"{group[0]}-{group[1]}B-{group[2]}")
        xs.append(sorteddf["num_peers"].tolist())
        ys.append(sorteddf["cpu_usage"].tolist())

    tikz_plot_cpu = TikzPlotGenerator(
        xs=xs,
        ys=ys,
        cat=cats,
        xlabel="Number of Peers",
        ylabel="CPU Usage",
        filename=f"outputs/{notebook_name}/modubft-spread-over-azs-cpu-usage-role{role}.tex"
    )
    tikz_plot_cpu.save()
    tikz_plot_cpu.compile(output_dir=f"outputs/{notebook_name}/")



[np.int64(3), np.int64(6), np.int64(12), np.int64(18), np.int64(21)]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode

(./outputs/01-c5.large-modubft-spread-over-azs/modubft-spread-over-azs-cpu-usag
e-role2.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf

In [70]:
# Calculate the number of messages for each PBFT phase per second
# PrePrepare: one per peer per operation
preprepare_messages = throughputs["throughput"] * throughputs["num_peers"]

# Prepare: each peer sends a prepare to every other peer (including itself)
prepare_messages = throughputs["throughput"] * throughputs["num_peers"] ** 2

# Commit: each peer sends a commit to every other peer except itself
commit_messages = throughputs["throughput"] * throughputs["num_peers"] * (throughputs["num_peers"] - 1)

# Client responses: each peer sends a response to the client
client_responses = throughputs["throughput"] * throughputs["num_peers"]

# Total messages per second (divided by 10 for normalization)
total_messages = (preprepare_messages + prepare_messages + commit_messages + client_responses) / 10

# Add results to the dataframe
throughputs["preprepare_messages"] = preprepare_messages
throughputs["prepare_messages"] = prepare_messages
throughputs["commit_messages"] = commit_messages
throughputs["client_responses"] = client_responses
throughputs["total_messages"] = total_messages

# Prepare data for plotting
xs, ys, cats = [], [], []
for (benchmark, vallen), df in throughputs.groupby(["benchmark", "vallen"]):
    cats.append(f"{benchmark}-{vallen}B")
    xs.append(df["num_peers"].tolist())
    ys.append(df["total_messages"].tolist())

# Plot using TikzPlotGenerator
tikz_plot_messages = TikzPlotGenerator(
    xs=xs,
    ys=ys,
    cat=cats,
    xlabel="Number of Peers",
    ylabel="Total Messages Sent (messages/sec)",
    filename=f"outputs/{notebook_name}/modubft-spread-over-azs-all-messages.tex"
)
tikz_plot_messages.save()
tikz_plot_messages.compile(output_dir=f"outputs/{notebook_name}/")



[np.int64(3), np.int64(6), np.int64(12), np.int64(18), np.int64(21)]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode

(./outputs/01-c5.large-modubft-spread-over-azs/modubft-spread-over-azs-all-mess
ages.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-di

0

In [71]:

preprepare_messages = throughputs["throughput"]*throughputs["num_peers"]
prepare_messages = throughputs["throughput"] *throughputs["num_peers"] 
commit_messages = throughputs["throughput"] *(throughputs["num_peers"]-1) 
client_responses = throughputs["throughput"]

total_messages = preprepare_messages + prepare_messages + commit_messages + client_responses

throughputs["preprepare_messages"] = preprepare_messages
throughputs["prepare_messages"] = prepare_messages
throughputs["commit_messages"] = commit_messages
throughputs["client_responses"] = client_responses
throughputs["total_messages"] = total_messages/10

# fig = go.Figure() 

# for data in throughputs.groupby("vallen"): 
#     vallen, df = data
#     fig.add_trace(go.Scatter(
#         x=df["num_peers"],
#         y=df["total_messages"],
#         mode="lines+markers",
#         name=f"Vallen={vallen}"
#     ))
# fig.update_layout(
#     title="Total Messages Sent vs Number of Peers",
#     xaxis_title="Number of Peers",
#     yaxis_title="Total Messages Sent (messages/sec)"
# )
# fig.show()
# tikz_plot_messages = TikzPlotGenerator(
#     xs=extract_groups_as_lists(throughputs,"num_peers","vallen"),
#     ys=extract_groups_as_lists(throughputs,"total_messages","vallen"),
#     cat=extract_unique_categories(throughputs,"vallen") ,
#     xlabel="Number of Peers",
#     ylabel="Total Messages Sent (messages/sec)",
#     filename=f"outputs/{notebook_name}/modubft-spread-over-azs-messages.tex"
# )
# tikz_plot_messages.save()
# tikz_plot_messages.compile(output_dir=f"outputs/{notebook_name}/")


xs = []
ys = []
cats = []

for (group, df) in throughputs.groupby(["benchmark", "vallen"]):
    cats.append(f"{group[0]}-{group[1]}B")
    xs.append(df["num_peers"].tolist())
    ys.append(df["total_messages"].tolist())

tikz_plot_messages = TikzPlotGenerator(
    xs=xs,
    ys=ys,
    cat=cats,
    xlabel="Number of Peers",
    ylabel="Total Messages Sent (messages/sec)",
    filename=f"outputs/{notebook_name}/modubft-spread-over-azs-messages.tex"
)
tikz_plot_messages.save()
tikz_plot_messages.compile(output_dir=f"outputs/{notebook_name}/")


[np.int64(3), np.int64(6), np.int64(12), np.int64(18), np.int64(21)]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode

(./outputs/01-c5.large-modubft-spread-over-azs/modubft-spread-over-azs-messages
.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-dist/t

0

In [72]:

bytes_sent = process_bytes_sent(benchmarks)
bytes_sent["benchmark"] = bytes_sent["benchmark"].apply(lambda s: "c5a.large" if "large" in s else "c5a.xlarge")

bytes_sent

,benchmark,num_peers,vallen,runtime,typ,bytes_sent,role,cluster_batch_size,number_of_clusters
0,c5a.large,3,512,None,Commit,55029200.0,2,1,1
1,c5a.large,3,512,None,Checkpoint,133212.0,2,1,1
2,c5a.large,3,512,None,PrePrepare,525263439.0,2,1,1
3,c5a.large,3,512,None,Prepare,66605286.0,2,1,1
4,c5a.large,3,512,None,ClientResponse,19303566.0,2,1,1
...,...,...,...,...,...,...,...,...,...
1345,c5a.xlarge,21,4096,None,Commit,8356390.3,1,1,1
1346,c5a.xlarge,21,4096,None,Checkpoint,16942.8,1,1,1
1347,c5a.xlarge,21,4096,None,PrePrepare,0.0,1,1,1
1348,c5a.xlarge,21,4096,None,Prepare,8591139.9,1,1,1


In [85]:
bytes_sent_in_gbps = (bytes_sent.groupby(["num_peers","vallen","role","benchmark"])[["bytes_sent"]].sum() * 8 * 10**(-9) * 15**(-1)).reset_index()

bytes_sent_role2 = bytes_sent_in_gbps.query("role == 2")

xs = []
ys = []
cats = []


for (group, df) in bytes_sent_role2.groupby(["vallen","benchmark"]):
    cats.append(f"{group[1]}-{group[0]}B")
    sorteddf = df.sort_values(["num_peers"])
    xs.append(sorteddf["num_peers"].tolist())
    ys.append(sorteddf["bytes_sent"].tolist())

tikz_plot_bytes_sent_role2 = TikzPlotGenerator(
    xs=xs,
    ys=ys,
    cat=cats,
    xlabel="Number of Peers",
    ylabel="Bytes Sent Percent",
    filename=f"outputs/{notebook_name}/modubft-spread-over-azs-bytes-sent-role2-gbps.tex"
)

tikz_plot_bytes_sent_role2.save()
tikz_plot_bytes_sent_role2.compile(output_dir=f"outputs/{notebook_name}/")

bytes_sent_role1 = bytes_sent_in_gbps.query("role == 1")
xs = []
ys = []
cats = []
for (group, df) in bytes_sent_role1.groupby(["vallen","benchmark"]):
    cats.append(f"{group[1]}-{group[0]}B")
    sorteddf = df.sort_values(["num_peers"])
    xs.append(sorteddf["num_peers"].tolist())
    ys.append(sorteddf["bytes_sent"].tolist())

tikz_plot_bytes_sent_role1 = TikzPlotGenerator(
    xs=xs,
    ys=ys,
    cat=cats,
    xlabel="Number of Peers",
    ylabel="Bytes Sent Percent",
    filename=f"outputs/{notebook_name}/modubft-spread-over-azs-bytes-sent-role1-gbps.tex"
)
tikz_plot_bytes_sent_role1.save()
tikz_plot_bytes_sent_role1.compile(output_dir=f"outputs/{notebook_name}/")




[np.int64(3), np.int64(6), np.int64(12), np.int64(18), np.int64(21)]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode

(./outputs/01-c5.large-modubft-spread-over-azs/modubft-spread-over-azs-bytes-se
nt-role2-gbps.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive

0

In [81]:


# # Example usage:
# role_df = get_role_df(bytes_sent, role=2)
# role_df_complete = complete_multiindex(role_df, ['num_peers', 'typ', 'vallen'])
# role_df_complete = add_combined_column(role_df_complete, ['num_peers', 'vallen'], 'num_peers_vallen')
# role_df_complete = add_percent_column(role_df_complete, 'num_peers_vallen', 'bytes_sent', 'bytes_sent_percent')

# plot_tikz(
#     role_df_complete,
#     x_col="num_peers_vallen",
#     y_col="bytes_sent_percent",
#     cat_col="typ",
#     filename=f"outputs/{notebook_name}/modubft-spread-over-azs-bytes-sent-role2.tex",
#     xlabel="(Number of Peers, Vallen)",
#     ylabel="Bytes Sent Percent",
#     bar=True,
#     stack=True,
#     symbolic_x=True,
#     sort_x_key=lambda x: eval(x),
#     output_dir=f"outputs/{notebook_name}/"
    
# )
# role_df = get_role_df(bytes_sent, role=1)
# role_df_complete = complete_multiindex(role_df, ['num_peers', 'typ', 'vallen'])
# role_df_complete = add_combined_column(role_df_complete, ['num_peers', 'vallen'], 'num_peers_vallen')
# role_df_complete = add_percent_column(role_df_complete, 'num_peers_vallen', 'bytes_sent', 'bytes_sent_percent')
# plot_tikz(
#     role_df_complete,
#     x_col="num_peers_vallen",
#     y_col="bytes_sent_percent",
#     cat_col="typ",
#     filename=f"outputs/{notebook_name}/modubft-spread-over-azs-bytes-sent-role1.tex",
#     xlabel="(Number of Peers, Vallen)",
#     ylabel="Bytes Sent Percent",
#     bar=True,
#     stack=True,
#     symbolic_x=True,
#     sort_x_key=lambda x: eval(x),
#     output_dir=f"outputs/{notebook_name}/"
# )
# For role 2
xs = []
ys = []
cats = []
role_df = get_role_df(bytes_sent, role=2)
role_df_complete = complete_multiindex(role_df, ['num_peers', 'typ', 'vallen', 'benchmark'])
role_df_complete = add_combined_column(role_df_complete, ['num_peers', 'vallen',"benchmark"], 'num_peers_vallen')
role_df_complete = add_percent_column(role_df_complete, 'num_peers_vallen', 'bytes_sent', 'bytes_sent_percent')

for (group, df) in role_df_complete.groupby(["typ"]):
    cats.append(f"{group[0]}")
    xs.append(df["num_peers_vallen"].tolist())
    ys.append(df["bytes_sent_percent"].tolist())

tikz_plot_bytes_sent_role2 = TikzPlotGenerator(
    xs=xs,
    ys=ys,
    cat=cats,
    xlabel="",
    ylabel="Bytes Sent Percent",
    filename=f"outputs/{notebook_name}/modubft-spread-over-azs-bytes-sent-role2-alt.tex",
    bar=True,
    stack=True,
    symbolic_x=True,
    sort_x_key=lambda x: (int(x.split(",")[0][1:]),int(x.split(",")[1]),str(x.split(",")[2:-1]))
)
tikz_plot_bytes_sent_role2.save()
tikz_plot_bytes_sent_role2.compile(output_dir=f"outputs/{notebook_name}/")

# For role 1
role_df = get_role_df(bytes_sent, role=1)
role_df_complete = complete_multiindex(role_df, ['num_peers', 'typ', 'vallen', 'benchmark'])
role_df_complete = add_combined_column(role_df_complete, ['num_peers', 'vallen',"benchmark"], 'num_peers_vallen')
role_df_complete = add_percent_column(role_df_complete, 'num_peers_vallen', 'bytes_sent', 'bytes_sent_percent')

xs = []
ys = []
cats = []

for (group, df) in role_df_complete.groupby(["typ"]):
    cats.append(f"{group[0]}")
    xs.append(df["num_peers_vallen"].tolist())
    ys.append(df["bytes_sent_percent"].tolist())

tikz_plot_bytes_sent_role1 = TikzPlotGenerator(
    xs=xs,
    ys=ys,
    cat=cats,
    xlabel="",
    ylabel="Bytes Sent Percent",
    filename=f"outputs/{notebook_name}/modubft-spread-over-azs-bytes-sent-role1-alt.tex",
    bar=True,
    stack=True,
    symbolic_x=True,
    sort_x_key=lambda x: (int(x.split(",")[0][1:]), int(x.split(",")[1]), str(x.split(",")[2:-1]))
)
tikz_plot_bytes_sent_role1.save()
tikz_plot_bytes_sent_role1.compile(output_dir=f"outputs/{notebook_name}/")



[np.str_('(3,512,c5a.large)'), np.str_('(3,512,c5a.xlarge)'), np.str_('(3,4096,c5a.large)'), np.str_('(3,4096,c5a.xlarge)'), np.str_('(6,512,c5a.large)'), np.str_('(6,512,c5a.xlarge)'), np.str_('(6,4096,c5a.large)'), np.str_('(6,4096,c5a.xlarge)'), np.str_('(12,512,c5a.large)'), np.str_('(12,512,c5a.xlarge)'), np.str_('(12,4096,c5a.large)'), np.str_('(12,4096,c5a.xlarge)'), np.str_('(18,512,c5a.large)'), np.str_('(18,512,c5a.xlarge)'), np.str_('(18,4096,c5a.large)'), np.str_('(18,4096,c5a.xlarge)'), np.str_('(21,512,c5a.large)'), np.str_('(21,512,c5a.xlarge)'), np.str_('(21,4096,c5a.large)'), np.str_('(21,4096,c5a.xlarge)')]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode

(./outputs/01-c5.large-modubft-spread-over-azs/modubft-spread-over-azs-bytes-se
nt-role2-alt.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standal

0

In [80]:
role_df_complete

,num_peers,typ,vallen,benchmark,bytes_sent,role,cluster_batch_size,number_of_clusters,num_peers_vallen,bytes_sent_percent
0,3,Checkpoint,512,c5a.large,137076.0,2,2,2,"(3,512,c5a.large)",0.000943
1,3,Checkpoint,512,c5a.xlarge,262036.0,2,2,2,"(3,512,c5a.xlarge)",0.000941
2,3,Checkpoint,4096,c5a.large,102712.0,2,2,2,"(3,4096,c5a.large)",0.000943
3,3,Checkpoint,4096,c5a.xlarge,197852.0,2,2,2,"(3,4096,c5a.xlarge)",0.000941
4,3,ClientResponse,512,c5a.large,20269570.0,2,2,2,"(3,512,c5a.large)",0.139384
...,...,...,...,...,...,...,...,...,...,...
95,21,PrePrepare,4096,c5a.xlarge,0.0,20,20,20,"(21,4096,c5a.xlarge)",0.000000
96,21,Prepare,512,c5a.large,92361206.0,20,20,20,"(21,512,c5a.large)",0.499228
97,21,Prepare,512,c5a.xlarge,231987645.6,20,20,20,"(21,512,c5a.xlarge)",0.496813
98,21,Prepare,4096,c5a.large,64766422.7,20,20,20,"(21,4096,c5a.large)",0.496350


In [88]:
throughputs = process_throughput_benchmarks(benchmarks)
res = {}
array_length = 1000
for benchmark in benchmarks: 

    received = np.zeros(array_length)
    response = np.zeros(array_length)
    for log in glob(f"{benchmark}/*.log"): 
        
        with open(log, 'r') as f:
            lines = [line for line in f.read().splitlines() if "client request" in line]

        if len(lines) != 2*array_length : 
            continue
        if lines : 
           
            for line in lines : 
                nanoseconds = line.split(" ")[-1]
                if "Received" in line: 
                    rc = int(line.split(" ")[3])
                    # print(line)
                    received[rc] = int(nanoseconds)
                    rc += 1
                elif "Response" in line: 
                    rp = int(line.split(" ")[3])
                    response[rp] = int(nanoseconds)
                    rp += 1
                else: 
                    raise ValueError("Unexpected line")
            
            latency = response - received
    latency_ms = latency * 1e-6
    max_latency = np.max(latency_ms)
    avg_latency = np.median(latency_ms)
    min_latency = np.min(latency_ms)
    throughputs.loc[throughputs['benchmark'] == benchmark, 'latency_ms'] = avg_latency
    throughputs.loc[throughputs['benchmark'] == benchmark, 'max_latency_ms'] = max_latency
    throughputs.loc[throughputs['benchmark'] == benchmark, 'min_latency_ms'] = min_latency
    
    if min_latency < 0 : 
        raise ValueError("Negative latency detected")
    print(f"Benchmark: {benchmark}, Max Latency: {max_latency}, Avg Latency: {avg_latency}, Min Latency: {min_latency}")
        

Benchmark: ../../aws/benchmark/out/modubft/20251025-215157-large/20251025215534-cb1-v512/, Max Latency: 34.218496, Avg Latency: 29.03744, Min Latency: 8.557824
Benchmark: ../../aws/benchmark/out/modubft/20251025-215157-large/20251025215609-cb1-v4096/, Max Latency: 45.059072, Avg Latency: 42.13248, Min Latency: 11.263743999999999
Benchmark: ../../aws/benchmark/out/modubft/20251025-215157-large/20251025220150-cb1-v512/, Max Latency: 60.7104, Avg Latency: 48.440576, Min Latency: 12.711167999999999
Benchmark: ../../aws/benchmark/out/modubft/20251025-215157-large/20251025220231-cb1-v4096/, Max Latency: 94.570752, Avg Latency: 80.883456, Min Latency: 28.014592
Benchmark: ../../aws/benchmark/out/modubft/20251025-215157-large/20251025220833-cb1-v512/, Max Latency: 108.962048, Avg Latency: 97.413376, Min Latency: 14.843648
Benchmark: ../../aws/benchmark/out/modubft/20251025-215157-large/20251025220922-cb1-v4096/, Max Latency: 162.49215999999998, Avg Latency: 154.031488, Min Latency: 83.55942399

In [90]:
throughputs["benchmark"] = throughputs["benchmark"].apply(lambda s: "c5a.large" if "large" in s else "c5a.xlarge")

xs = []
ys = []
cats = []
for (group, df) in throughputs.groupby(["benchmark", "vallen"]):
    cats.append(f"{group[0]}-{group[1]}B")
    xs.append(df["num_peers"].tolist())
    ys.append(df["latency_ms"].tolist())
    
tikz_plot = TikzPlotGenerator(
    xs=xs,
    ys=ys,
    cat=cats,
    xlabel="Number of Peers",
    ylabel="Latency (ms)",
    filename=f"outputs/{notebook_name}/modubft-spread-over-azs-latency.tex"
)
tikz_plot.save()
tikz_plot.compile(output_dir=f"outputs/{notebook_name}/")



# plot_tikz(
#     throughputs,
#     y_col="latency_ms",
#     x_col="num_peers",
#     cat_col="vallen",
#     filename=f"outputs/{notebook_name}/modubft-spread-over-azs-latency.tex",
#     xlabel="Number of Peers",
#     ylabel="Latency (ms)",
#     output_dir=f"outputs/{notebook_name}/"
# )

[np.int64(3), np.int64(6), np.int64(12), np.int64(18), np.int64(21)]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode

(./outputs/01-c5.large-modubft-spread-over-azs/modubft-spread-over-azs-latency.
tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-dist/te

0

In [92]:
#rename every file in output_dir=f"outputs/{notebook_name}/"
import os
output_dir=f"outputs/{notebook_name}/"
for filename in os.listdir(output_dir):
    if filename.endswith(".tex") or filename.endswith(".pdf"):
        new_filename = filename.replace("modubft-spread-over-azs", "modubft-c5a-large-xlarge-spread-over-azs")
        os.rename(os.path.join(output_dir, filename), os.path.join(output_dir, new_filename))
        
# remove log and aux 
for filename in os.listdir(output_dir):
    if filename.endswith(".log") or filename.endswith(".aux"):
        os.remove(os.path.join(output_dir, filename))
        